# DCA Ingest Gantenbein (Separated TTL Outputs)

This notebook generates **separate TTL files** for:
- DROID base metadata
- EXIF/XMP enrichment delta
- PROV events and agents

`enriched.ttl` is intentionally not generated in this workflow.

In [2]:
from pathlib import Path
from urllib.parse import quote
import json
import subprocess
import hashlib
import shutil
import pandas as pd
from rdflib import Graph, Namespace, URIRef, Literal, BNode
from rdflib.namespace import RDF, RDFS, XSD, DCTERMS

In [3]:
# Paths
BASE = Path('/home/renku/work/dcaonnextcloud-500gb')
SOURCE_DIR = BASE / 'DigitalMaterialCopies/WeingutGantenbein/gramazio-kohler-archiv-server'
RESULT_DIR = BASE / 'dca-metadataraw/WeingutGantenbein/gramazio-kohler-archiv-server_results'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

NEXTCLOUD_DAV_BASE = 'https://nextcloud.ethz.ch/remote.php/dav/files/padrian/DCA'

DROID_CSV = RESULT_DIR / 'gramazio-kohler-archiv-server_DROIDresults.csv'
OUT_DROID = RESULT_DIR / '036_WeingutGantenbein_DROID.ttl'
OUT_EXIFXMP = RESULT_DIR / '036_WeingutGantenbein_EXIFXMP.ttl'
OUT_PROV = RESULT_DIR / '036_WeingutGantenbein_prov.ttl'

EXIFTOOL = 'exiftool'
XMP_JSON = RESULT_DIR / 'gramazio-kohler-archiv-server_xmp.json'

print('SOURCE_DIR:', SOURCE_DIR)
print('DROID_CSV :', DROID_CSV)
print('OUT_DROID :', OUT_DROID)
print('OUT_EXIFXMP:', OUT_EXIFXMP)
print('OUT_PROV  :', OUT_PROV)
print('NEXTCLOUD_DAV_BASE:', NEXTCLOUD_DAV_BASE)
print('EXIFTOOL:', EXIFTOOL)
print('SYSTEM_EXIFTOOL:', shutil.which(EXIFTOOL))

SOURCE_DIR: /home/renku/work/dcaonnextcloud-500gb/DigitalMaterialCopies/WeingutGantenbein/gramazio-kohler-archiv-server
DROID_CSV : /home/renku/work/dcaonnextcloud-500gb/dca-metadataraw/WeingutGantenbein/gramazio-kohler-archiv-server_results/gramazio-kohler-archiv-server_DROIDresults.csv
OUT_DROID : /home/renku/work/dcaonnextcloud-500gb/dca-metadataraw/WeingutGantenbein/gramazio-kohler-archiv-server_results/036_WeingutGantenbein_DROID.ttl
OUT_EXIFXMP: /home/renku/work/dcaonnextcloud-500gb/dca-metadataraw/WeingutGantenbein/gramazio-kohler-archiv-server_results/036_WeingutGantenbein_EXIFXMP.ttl
OUT_PROV  : /home/renku/work/dcaonnextcloud-500gb/dca-metadataraw/WeingutGantenbein/gramazio-kohler-archiv-server_results/036_WeingutGantenbein_prov.ttl
NEXTCLOUD_DAV_BASE: https://nextcloud.ethz.ch/remote.php/dav/files/padrian/DCA
EXIFTOOL: exiftool
SYSTEM_EXIFTOOL: /layers/paketo-buildpacks_conda-env-update/conda-env/bin/exiftool


In [5]:
# Namespaces
DCA = Namespace('http://dca.ethz.ch/ontology#')
DCA_ID = Namespace('http://dca.ethz.ch/id/')
PREMIS = Namespace('http://www.loc.gov/premis/rdf/v3/')
RICO = Namespace('https://www.ica.org/standards/RiC/ontology#')

def make_graph():
    g = Graph()
    g.bind('dca', DCA)
    g.bind('dca-id', DCA_ID)
    g.bind('premis', PREMIS)
    g.bind('rico', RICO)
    g.bind('dcterms', DCTERMS)
    g.bind('xsd', XSD)
    return g

def file_uri_from_md5(md5_value: str, fallback_path: str) -> URIRef:
    if isinstance(md5_value, str) and md5_value.strip():
        return URIRef(f'http://dca.ethz.ch/id/file_{md5_value.strip()[:16]}')
    digest = hashlib.sha1(str(fallback_path).encode('utf-8')).hexdigest()
    return URIRef(f'http://dca.ethz.ch/id/file_{digest[:16]}')

def local_path_to_nextcloud_uri(path_value: str):
    if not isinstance(path_value, str) or not path_value.strip():
        return None
    try:
        rel = Path(path_value).relative_to(BASE)
    except Exception:
        return None
    encoded_rel = '/'.join(quote(part, safe='') for part in rel.parts)
    return URIRef(f"{NEXTCLOUD_DAV_BASE}/{encoded_rel}")

def add_identifier(g: Graph, file_uri: URIRef, id_type: str, id_value: str):
    if not id_value:
        return
    b = BNode()
    g.add((file_uri, PREMIS.hasIdentifier, b))
    g.add((b, PREMIS.identifierType, Literal(id_type)))
    g.add((b, PREMIS.identifierValue, Literal(str(id_value))))

## 1) DROID CSV -> DROID TTL (base graph only)

In [6]:
droid_df = pd.read_csv(DROID_CSV, low_memory=False)
g_droid = make_graph()
project_uri = URIRef('http://dca.ethz.ch/id/project_WeingutGantenbein')

for _, row in droid_df.iterrows():
    file_path = str(row.get('FILE_PATH', ''))
    file_name = str(row.get('NAME', '') or Path(file_path).name)
    md5_hash = str(row.get('HASH', '') or '')

    file_uri = file_uri_from_md5(md5_hash, file_path)
    g_droid.add((file_uri, RDF.type, DCA.ArchiveFile))
    g_droid.add((file_uri, RDF.type, PREMIS.Object))
    g_droid.add((file_uri, RDF.type, RICO.Record))
    g_droid.add((file_uri, DCA.belongsToProject, project_uri))

    if file_path:
        nextcloud_uri = local_path_to_nextcloud_uri(file_path)
        if nextcloud_uri is not None:
            g_droid.add((file_uri, DCTERMS.identifier, nextcloud_uri))
        else:
            g_droid.add((file_uri, DCTERMS.identifier, Literal(file_path)))
    if file_name:
        g_droid.add((file_uri, DCTERMS.title, Literal(file_name)))

    modified = row.get('LAST_MODIFIED')
    if isinstance(modified, str) and modified:
        g_droid.add((file_uri, DCTERMS.modified, Literal(modified, datatype=XSD.dateTime)))

    fmt_name = row.get('FORMAT_NAME')
    if isinstance(fmt_name, str) and fmt_name:
        g_droid.add((file_uri, PREMIS.hasFormatName, Literal(fmt_name)))

    mime = row.get('MIME_TYPE')
    puid = row.get('PUID')
    if isinstance(mime, str) and mime:
        g_droid.add((file_uri, PREMIS.hasFormatNote, Literal(f'MIME: {mime}')))
    if isinstance(puid, str) and puid:
        g_droid.add((file_uri, PREMIS.hasFormatNote, Literal(f'PRONOM ID: {puid}')))

    size = row.get('SIZE')
    if pd.notna(size):
        g_droid.add((file_uri, PREMIS.hasSize, Literal(int(size), datatype=XSD.long)))

    g_droid.add((file_uri, PREMIS.hasCreatingApplication, Literal('DROID: Signature')))
    add_identifier(g_droid, file_uri, 'MD5 Hash', md5_hash if md5_hash else '')

g_droid.serialize(OUT_DROID, format='turtle')
print('DROID triples:', len(g_droid))
print('Saved:', OUT_DROID)

DROID triples: 211352
Saved: /home/renku/work/dcaonnextcloud-500gb/dca-metadataraw/WeingutGantenbein/gramazio-kohler-archiv-server_results/036_WeingutGantenbein_DROID.ttl


## 2) EXIF/XMP -> EXIFXMP TTL (delta only)

In [7]:
if shutil.which(EXIFTOOL) is None:
    raise RuntimeError(
        "ExifTool ist nicht installiert. Bitte einmalig in Renku ausführen:\n"
        "micromamba install -y -c conda-forge exiftool"
    )

cmd = [EXIFTOOL, '-json', '-r', str(SOURCE_DIR)]
result = subprocess.run(cmd, capture_output=True, text=True, check=True)
XMP_JSON.write_text(result.stdout, encoding='utf-8')
xmp_rows = json.loads(result.stdout)

g_xmp = make_graph()

for rec in xmp_rows:
    src = str(rec.get('SourceFile', ''))
    md5 = str(rec.get('MD5Digest', '') or rec.get('MD5', '') or '')
    file_uri = file_uri_from_md5(md5, src)

    g_xmp.add((file_uri, RDF.type, DCA.ArchiveFile))

    app = rec.get('CreatorTool') or rec.get('Software') or rec.get('Application')
    if isinstance(app, str) and app:
        g_xmp.add((file_uri, PREMIS.hasCreatingApplication, Literal(app)))

    docid = rec.get('DocumentID') or rec.get('XMP:DocumentID')
    instid = rec.get('InstanceID') or rec.get('XMP:InstanceID')

    if isinstance(docid, str) and docid:
        add_identifier(g_xmp, file_uri, 'XMP DocumentID', docid)
    if isinstance(instid, str) and instid:
        add_identifier(g_xmp, file_uri, 'XMP InstanceID', instid)

g_xmp.serialize(OUT_EXIFXMP, format='turtle')
print('EXIF/XMP triples:', len(g_xmp))
print('Saved:', OUT_EXIFXMP)

EXIF/XMP triples: 39044
Saved: /home/renku/work/dcaonnextcloud-500gb/dca-metadataraw/WeingutGantenbein/gramazio-kohler-archiv-server_results/036_WeingutGantenbein_EXIFXMP.ttl


## 3) Provenance TTL (events and agents only)

In [8]:
g_prov = make_graph()
activity_droid = URIRef('http://dca.ethz.ch/id/activity_DROID_Conversion_2026')
activity_xmp = URIRef('http://dca.ethz.ch/id/activity_EXIFXMP_Integration_2026')
agent_droid = URIRef('http://dca.ethz.ch/id/agent_DROID')
agent_exiftool = URIRef('http://dca.ethz.ch/id/agent_ExifTool')

g_prov.add((activity_droid, RDF.type, PREMIS.Event))
g_prov.add((activity_droid, RDFS.label, Literal('DROID to RDF Conversion')))
g_prov.add((activity_droid, DCTERMS.date, Literal('2026-03-31', datatype=XSD.date)))

g_prov.add((activity_xmp, RDF.type, PREMIS.Event))
g_prov.add((activity_xmp, RDFS.label, Literal('EXIF/XMP Integration')))
g_prov.add((activity_xmp, DCTERMS.date, Literal('2026-03-31', datatype=XSD.date)))

g_prov.add((agent_droid, RDF.type, PREMIS.Agent))
g_prov.add((agent_droid, RDFS.label, Literal('DROID')))

g_prov.add((agent_exiftool, RDF.type, PREMIS.Agent))
g_prov.add((agent_exiftool, RDFS.label, Literal('ExifTool')))

g_prov.add((activity_droid, PREMIS.hasAgent, agent_droid))
g_prov.add((activity_xmp, PREMIS.hasAgent, agent_exiftool))

g_prov.serialize(OUT_PROV, format='turtle')
print('PROV triples:', len(g_prov))
print('Saved:', OUT_PROV)

PROV triples: 12
Saved: /home/renku/work/dcaonnextcloud-500gb/dca-metadataraw/WeingutGantenbein/gramazio-kohler-archiv-server_results/036_WeingutGantenbein_prov.ttl


## 4) Quick validation

In [9]:
print('Exists DROID  :', OUT_DROID.exists(), OUT_DROID)
print('Exists EXIFXMP:', OUT_EXIFXMP.exists(), OUT_EXIFXMP)
print('Exists PROV   :', OUT_PROV.exists(), OUT_PROV)

# Guardrails: PROV should not carry DROID signature payload
prov_text = OUT_PROV.read_text(encoding='utf-8') if OUT_PROV.exists() else ''
print('PROV contains DROID signature?', 'DROID: Signature' in prov_text)

# Guardrails: DROID should not carry XMP DocumentID payload
droid_text = OUT_DROID.read_text(encoding='utf-8') if OUT_DROID.exists() else ''
print('DROID contains XMP DocumentID?', 'XMP DocumentID' in droid_text)

Exists DROID  : True /home/renku/work/dcaonnextcloud-500gb/dca-metadataraw/WeingutGantenbein/gramazio-kohler-archiv-server_results/036_WeingutGantenbein_DROID.ttl
Exists EXIFXMP: True /home/renku/work/dcaonnextcloud-500gb/dca-metadataraw/WeingutGantenbein/gramazio-kohler-archiv-server_results/036_WeingutGantenbein_EXIFXMP.ttl
Exists PROV   : True /home/renku/work/dcaonnextcloud-500gb/dca-metadataraw/WeingutGantenbein/gramazio-kohler-archiv-server_results/036_WeingutGantenbein_prov.ttl
PROV contains DROID signature? False
DROID contains XMP DocumentID? False
